# Predict a number from a CSV with Python

**Your first AutoML regression model: concrete strength**

Imagine you have a spreadsheet of concrete test results. Each row describes
the ingredients, the specimen's age, and its measured strength. For another
mixture, you know the ingredients and age but want an estimate of strength.
Can a model learn from those past measurements to make that prediction?

Predicting a number is called **regression**. In this notebook, you will look
at the data, train several models with AutoML, and compare their predictions
with measurements that were kept out of training. Then you will read the
reports and reload the saved model. Basic familiarity with running Python
is enough to begin; we explain the machine-learning concepts as we use them.

## A first look at the data

These are the first five rows of the real CSV, showing four of its nine columns:

| Cement (kg/m³) | Water (kg/m³) | Age (days) | Measured strength (MPa) |
| ---: | ---: | ---: | ---: |
| 540.0 | 162.0 | 28 | 79.99 |
| 540.0 | 162.0 | 28 | 61.89 |
| 332.5 | 228.0 | 270 | 40.27 |
| 332.5 | 228.0 | 365 | 41.05 |
| 198.6 | 192.0 | 360 | 44.30 |

The first row describes a mixture with 540 kg of cement and 162 kg of water
per cubic metre, tested at 28 days, with measured strength 79.99 MPa.
The other five ingredient columns are omitted from this short preview;
for example, the first two rows have different coarse aggregate amounts.
The full table appears after loading below.

We give the model eight **inputs**, also called *features*: seven ingredient
quantities and age. We want it to predict one **target**: strength in MPa
(megapascals). Do not include that measured target among the prediction inputs.

## Set up your notebook

Use Python 3.11 or 3.12. Before starting this notebook, install into its Python
environment: `python -m pip install 'mljar-supervised==1.4.0b1' jupyterlab ipywidgets`.
Start Jupyter with `python -m jupyterlab`. No GPU or optional Torch packages
are needed. The UCI CSV download is 41.5 kB; package installation is larger.

This notebook uses a 120-second training budget and two CPU threads. Training
can finish early; total runtime also includes imports, reports, and reload.
Script runs on the recorded Linux server took 16–21 seconds. Laptop and 8 GB
measurements are still pending. Your environment can produce different timings.

Run cells in order. Use a new output directory for each run. The code below
is generated from `examples/scripts/concrete_strength.py`; it is fully
included here, so you do not need that file to execute this notebook.


## 1. Load and inspect the measurements

One row contains seven ingredient amounts (kg/m³), specimen age (days), and
measured compressive strength (MPa). The **target** is the quantity to predict:
`Concrete compressive strength`. All other columns are numeric inputs.

Source: Yeh, I. (1998), *Concrete Compressive Strength*,
[UCI Machine Learning Repository](https://doi.org/10.24432/C5PK67),
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
We download UCI's CSV unchanged and check the reviewed file's checksum.
The 1,030 rows contain no missing values. Group labels and predictions are
additions made by this example. A changed checksum stops the example so a
changed source cannot silently alter the results.


In [ ]:
import argparse
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
import textwrap
import time
from pathlib import Path
from urllib.request import urlopen

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from threadpoolctl import threadpool_limits

from supervised import AutoML


DATA_URL = "https://archive.ics.uci.edu/static/public/165/data.csv"
DATA_SHA256 = "8d4b15b6fc68cd932d745cbd663d5ceae66dd54422e99c1e4865f2936ab7e2af"
INGREDIENTS = [
    "Cement", "Blast Furnace Slag", "Fly Ash", "Water", "Superplasticizer",
    "Coarse Aggregate", "Fine Aggregate",
]
FEATURES = INGREDIENTS + ["Age"]
TARGET = "Concrete compressive strength"
SEED = 42


def load_data(path):
    """Download once, then check the exact UCI snapshot on every run."""
    if not path.exists():
        with urlopen(DATA_URL, timeout=60) as response:
            content = response.read()
        if hashlib.sha256(content).hexdigest() != DATA_SHA256:
            raise ValueError("UCI data changed; review the new snapshot before using it.")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != DATA_SHA256:
        raise ValueError("Dataset checksum differs from the reviewed UCI snapshot.")
    data = pd.read_csv(path)
    if list(data.columns) != FEATURES + [TARGET] or data.shape != (1030, 9):
        raise ValueError("Unexpected concrete dataset schema.")
    if not np.isfinite(data.to_numpy(dtype=float)).all():
        raise ValueError("The reviewed dataset should contain only finite numbers.")
    return data



from IPython.display import display


In [ ]:
output = Path("AutoML_ConcreteStrength_Notebook").resolve()
data_path = Path("AutoML_ConcreteData/concrete.csv").resolve()
budget = 120
thread_limits = threadpool_limits(limits=2)
# Step 1. Load the measurements and choose inputs and target.
# Seven ingredient quantities (kg/m³) and age (days) predict strength (MPa).
started = time.perf_counter()
output.mkdir(parents=True, exist_ok=False)
data = load_data(data_path)
X, y = data[FEATURES], data[TARGET]
display(data.head())


## 2. Reserve mixtures for testing

Some mixtures were measured at multiple ages. A random row split could put
the same recipe in training and testing. Here we ask about **unseen mixtures**,
so we group rows by the seven ingredient amounts and keep each group together.

We reserve 86 groups (192 rows) for final testing. The remaining 341 groups
(838 rows) train the models. Within training, three grouped folds rotate which
rows train and validate each candidate. AutoML uses those validation results
to select a model. The held-out test target is used only for final evaluation.

These exact ingredient matches are proxy groups, not verified experiment IDs.
Near-identical recipes or shared laboratories may still cross the split.
No batch/site identifiers are available to check those relationships.


In [ ]:
# Keep the same mixture at different ages on one side of every split (step 2).
# These are ingredient-based proxy groups, not verified experiment IDs.
groups = data.groupby(INGREDIENTS, sort=True).ngroup()
train, test = next(GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=SEED
).split(X, y, groups))
X_train, y_train = X.iloc[train].reset_index(drop=True), y.iloc[train].reset_index(drop=True)
X_test, y_test = X.iloc[test], y.iloc[test]
train_groups = groups.iloc[train].reset_index(drop=True)
cv = list(GroupKFold(n_splits=3).split(X_train, y_train, train_groups))
assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
for fit_rows, valid_rows in cv:
    assert set(train_groups.iloc[fit_rows]).isdisjoint(train_groups.iloc[valid_rows])

split_record = {
    "seed": SEED, "train_rows": train.tolist(), "test_rows": test.tolist(),
    "mixture_groups": groups.tolist(),
    "cv_indices_relative_to_train": [
        {"train": a.tolist(), "validation": b.tolist()} for a, b in cv
    ],
}
(output / "splits.json").write_text(json.dumps(split_record, indent=2) + "\n")
print(f"Concrete: {len(data)} measurements, {groups.nunique()} mixture groups.", flush=True)
print(f"Train: {len(train)} rows; held-out test: {len(test)} rows. Target: MPa.", flush=True)
print(data.head().to_string(index=False), flush=True)


## 3. Train a baseline and compare models

A baseline always predicts the median training strength. The median minimizes
constant absolute error, giving us a useful reference for **MAE**, the mean
absolute difference between a prediction and its measured value.

AutoML compares Baseline, Linear, Decision Tree, and LightGBM using the three
training folds. It also tries an ensemble that combines candidate predictions.
`n_jobs=2` limits model parallelism, and `total_time_limit=120` assigns a
training budget. Keep the seed and split fixed when comparing configurations.

Feature discovery is included in the base package, but currently does not
support custom grouped validation, so it is disabled here. Other feature
engineering and stacking are also disabled to keep this first run focused.
The `Baseline` in AutoML's leaderboard uses its own implementation; the
separately fitted median is our held-out evaluation reference.


In [ ]:
# A training median is our baseline (step 3): the test target is unused.
# AutoML compares models using only the three training-validation folds.
baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
automl = AutoML(
    results_path=str(output / "automl"), mode="Explain",
    algorithms=["Baseline", "Linear", "Decision Tree", "LightGBM"],
    ml_task="regression", eval_metric="mae", total_time_limit=budget,
    validation_strategy={"validation_type": "custom"},
    feature_discovery=False, golden_features=False, kmeans_features=False,
    features_selection=False, stack_models=False, train_ensemble=True,
    explain_level=1, n_jobs=2, random_state=SEED,
)
fit_started = time.perf_counter()
automl.fit(X_train, y_train, cv=cv)
fit_seconds = time.perf_counter() - fit_started


## 4. Evaluate on the held-out rows and predict example inputs

These rows were not used for training or model selection. We save predictions
with their measurements for inspection. The three example inputs are held-out
rows with the target removed, not proposed concrete designs or new experiments.


In [ ]:
# Step 4. Evaluate on rows excluded from training and model selection.
predictions = automl.predict(X_test)
baseline_predictions = baseline.predict(X_test)
print(f"Baseline test MAE: {mean_absolute_error(y_test, baseline_predictions):.2f} MPa", flush=True)
print(f"AutoML test MAE: {mean_absolute_error(y_test, predictions):.2f} MPa", flush=True)
X_test.assign(
    source_row=test, measured_strength_mpa=y_test,
    predicted_strength_mpa=predictions, baseline_strength_mpa=baseline_predictions,
).to_csv(output / "test_predictions.csv", index=False)

# Demonstrate new, unlabeled inputs using held-out examples (not invented designs).
new_rows = X_test.head(3)
new_rows.to_csv(output / "new_rows.csv", index=False)
new_predictions = automl.predict(new_rows)
print("Example input predictions (MPa):", new_predictions, flush=True)


The recorded beta script runs produced **4.30 MPa MAE**, compared with
**12.59 MPa** for the training-median baseline. A 4.30 MPa MAE means an average
absolute error of 4.30 MPa across these test rows; it does not bound every error.
Whether this is useful depends on the intended use and acceptable errors.
Mixtures with more measured rows contribute more to this row-weighted metric.

Do not repeatedly tune against this test result. Use training validation for
changes and reserve fresh evaluation data for a new final assessment.


## 5. Reload in a fresh Python process

Keep the entire `automl/` directory, including preprocessing and all ensemble
dependencies. This check starts a separate interpreter, reads the saved model
and input CSV, then verifies that predictions agree with the in-memory model.
It raises an error if they differ beyond a small numerical tolerance.


In [ ]:
# A new interpreter loads only the saved model directory and input CSV (step 5).
# This checks that future predictions work without our training-session state.
X_test.to_csv(output / "reload_inputs.csv", index=False)
subprocess.run([
    sys.executable, "-c",
    textwrap.dedent("""
    import sys
    import numpy as np
    import pandas as pd
    from supervised import AutoML

    model = AutoML(results_path=sys.argv[1])
    inputs = pd.read_csv(sys.argv[2])
    np.save(sys.argv[3], model.predict(inputs))
    """),
    str(output / "automl"), str(output / "reload_inputs.csv"),
    str(output / "reloaded_predictions.npy"),
], check=True)
reloaded = np.load(output / "reloaded_predictions.npy")
np.testing.assert_allclose(predictions, reloaded, rtol=1e-7, atol=1e-7)
print("Fresh-process reload agrees with the original predictions.", flush=True)


## 6. Inspect errors and save the run record

Points near the diagonal have predictions close to measured strength. Look
for large errors and ranges where predictions tend to be too high or low.
The example saves the figure, metrics, split indices, and package versions.
These records identify the run behind any reported number or chart.


In [ ]:
# Step 6. Plot errors and save the evaluation and environment record.
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
ax.scatter(y_test, predictions, s=20, alpha=0.65)
low, high = min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())
ax.plot([low, high], [low, high], color="black", linestyle="--", label="Perfect prediction")
ax.set(xlabel="Measured strength (MPa)", ylabel="Predicted strength (MPa)",
       title="Concrete strength: held-out mixture groups")
ax.legend()
fig.savefig(output / "predicted-vs-measured.png", dpi=160)
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(output / "predicted-vs-measured.png")))


In [ ]:
metrics = {
    "dataset_url": DATA_URL, "dataset_sha256": DATA_SHA256,
    "train_rows": len(train), "test_rows": len(test),
    "train_groups": int(groups.iloc[train].nunique()),
    "test_groups": int(groups.iloc[test].nunique()),
    "baseline_strategy": "training median",
    "baseline_mae_mpa": mean_absolute_error(y_test, baseline_predictions),
    "automl_mae_mpa": mean_absolute_error(y_test, predictions),
    "training_budget_seconds": budget, "fit_wall_seconds": fit_seconds,
    "workflow_seconds_excluding_imports": time.perf_counter() - started,
    "reload_max_absolute_difference": float(np.max(np.abs(predictions - reloaded))),
    "python": platform.python_version(), "platform": platform.platform(),
    "package_versions": {name: importlib.metadata.version(name) for name in (
        "mljar-supervised", "numpy", "pandas", "scikit-learn", "lightgbm",
        "features-goldmine", "joblib",
    )},
}
(output / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
print(json.dumps(metrics, indent=2), flush=True)
thread_limits.restore_original_limits()


## 7. Read the reports

Run the next cell to display the interactive HTML report here. Click a model
name to open its individual report and **Go back** to return to the overview.
This also writes `AutoML_ConcreteStrength_Notebook/automl/README.html`, which
you can open in a browser. The Markdown version is available as `README.md`.
The leaderboard compares **training validation** MAE, not the separate test
score above. Lower MAE is better. In the recorded run the selected ensemble
had validation MAE 4.66 MPa and test MAE 4.30 MPa, measured on different rows.

Follow the `4_Default_LightGBM` link to see its settings, validation metrics,
learning curves, permutation importance, and residual plots. Importance
describes model dependence on inputs; it does not establish causal effects.
The exact artifacts available depend on the algorithm and explanation level.


In [ ]:
# Step 7. Create the HTML report and print its browser location.
report = automl.report()
print(f"Open the run report: {(output / 'automl' / 'README.html').as_uri()}", flush=True)
print("Keep the complete automl directory for future predictions.", flush=True)
display(report)


## 8. Adapt this workflow to your CSV

1. Replace `load_data(data_path)` with `pd.read_csv("your_data.csv")`; the UCI
   checksum and schema checks apply only to the concrete dataset.
2. Set `TARGET` to your measured outcome and `FEATURES` to inputs available
   when predicting. For this regression example, the outcome must be numeric.
3. Replace `INGREDIENTS` and the grouping rule with your real experiment,
   customer, batch, or subject identifier. Keep identifiers out of model
   inputs unless their use is justified. Time-dependent data needs a time
   split; do not blindly reuse this mixture split.
4. Choose an untouched test set, a baseline, a budget, and an error measure
   meaningful in your units. Set a new output path before running.
5. Inspect errors, read reports, and verify prediction/reload with the same
   input columns and units. Domain knowledge determines acceptable errors.

To predict a later CSV in a new session:

```python
import pandas as pd
from supervised import AutoML
model = AutoML(results_path="AutoML_ConcreteStrength_Notebook/automl")
new_rows = pd.read_csv("AutoML_ConcreteStrength_Notebook/new_rows.csv")
predictions_mpa = model.predict(new_rows)
```

If the output directory already exists, choose a new name to retrain or load
the existing model as above. If a notebook cell fails, fix the cause, restart
the kernel, choose a new output directory, and run from the top.

**Limits:** exact recipe grouping cannot establish independence by experiment
or laboratory. A single split does not establish general performance.
Observational predictions do not prove that changing an ingredient causes
a strength change or replace physical testing and engineering assessment.

**Studio starting prompt (not yet validated in Studio):** “Help me run this
concrete-strength notebook. Explain the target and mixture groups, compare
the held-out error with the baseline, and show the saved model report.”
The Python/Jupyter workflow is tested separately; Studio execution remains
part of the release checklist.
